The following exercises are meant to be solved by gathering the bash commands incrimentally in two scripts, one for ex 1.* the other for ex 2.* 

### Ex 1

1\.a Make a new directory called `students` in your home. Download a csv file with the list of students of this lab from [here](https://www.dropbox.com/s/867rtx3az6e9gm8/LCP_22-23_students.csv) (use the `wget` command) and copy that to `students`. First check whether the file is already there

1\.b Make two new files, one containing the students belonging to PoD, the other to Physics.

1\.c For each letter of the alphabet, count the number of students whose surname starts with that letter. 

1\.d Find out which is the letter with most counts.

1\.e Assume an obvious numbering of the students in the file (first line is 1, second line is 2, etc.), group students "modulo 18", i.e. 1,19,37,.. 2,20,38,.. etc. and put each group in a separate file  

In [ ]:
# GENERAL RULES AND COMMENTS:

# SPACING IS ALWAYS WRONG IN BASH, UNLESS IT'S NECESSARY

# BASH OUTPUT REDIRECTION:
# In Bash there exist 3 File Descriptors or FD: FD0 (input) FD1 (main output) FD2 (error output)
# their names are FD0 standard input (strin), FD1 standard output (strout), FD3 standard error (strerr)
# > allows to redirect output (and < the input )
# command > file redirect output of command to file
# command 1> file is equivalent
# command 2> file redirect the error output to file
# command 1> file 2>&1, from right to left, redirect error copy to the std out and then the output to file, getting both out and err in file
# using > recreate output file, deleting pre existing content
# using >> appends the output to whatever already is present inside of file

# BASH CONCATENATE OPERATIONS: Piping as it's called
# to concatenate operation various structures can be used; the most common being |
# command_1 | command_2 sintax passes output of command_1 as input for command_2
# this concatenation is always performed left to right, and can be concatenated even more
# command_1 -options_1 input_file | command_2 -options_2 | command_3 -options_3 > output_file
# the previous perform the first operation on the input, passes the result to the second block, perform second operation using that as input,
# the second output is passed to the third block, and its output is then printed to the output_file

# COMMAND SUBSTITUTION: Getting the otuput of a command instead of the command
# Command substitution allows the output of a command to replace the command itself. 
# Command substitution occurs when a command is enclosed in
# `command`or $(command), $(( command )) for arithmetic operations
# of the two, only $(command) allows concatenations or nested operations
# a note, $(cat file) or `cat file` here used works but the fastest syntax is $(< file)

# HEREDOC or HereDoc(ument) notation: <<
# this can be used to litteraly write the input document runtime
# but if an additional < is added, it redirect the input to an actual document (strange but used by some command like read)

# bc comments
# scale must be specified to perform floating point operations, otherwise are integers
# echo 'scale=6;100*sqrt(3)/2.0' | bc

# THE EXTREME POWERFUL AWK: "Conosci awk? è una figata!" P. Ronchese
# awk is a programming language for processing text-based data
# it can be used to perform most of the requests managed by standard bash using a more classical langueage
# it is a sort of cheat code to solve these exercises, as in practice another language is called by bash
# awk performes a code on an input file with the basic structure
# awk '{program}' input_file
# awk works using FIELDS variables, with the default separator between fields in each row being ' '
# for each row, each field is accessible by its positional number ( $1, $2, ... ) and the $0 means "all fields"
# the prorgram is performed somehow vectorially, element-wise operating in each row at a time
# some option can be given to awk, the main one is -F that allows to change the field separator used -F',' is a tipical option
# conditions are put before the main body operation:
# awk '$9 == 10 { print $0}' file 
# print the whole line ($0) if the condition is mathced, the 9th field equal to the value of N varible
# awk '/biz/ {print $3}' filename
# print the third element in each row of filename which contains biz keyword, assuming fields in filename are separated by ' '
# the default command if nothing is provided as action in the body is { print $0 }, printing the whole line

# ADVANCED AWK
# the conditional presence of multiple elements in a row can be given using |, 
# /par_a|par_b|par_c/ {} makes the body execute only on rows containing all three parameters
# exit ends the "cycle" and stop the recursive call of body over rows when first met, allowing a "print only the first time.." situation
# using END before a body modifies the execution time, calling it only at the end of the document
# using BEGIN allows it to be executed before the first row is written, allowing a no input usage of awk
# variables from bash can be passed to awk using -v option
# awk -v N=5 '$9 == N { print $0}' file     noticing that N does not need $ after this
# awk -v N=$N '$9 == N { print $0}' file    if N was already defined in bash
# the NR parameter can be set to allow only a specific row to be read
# awk NR==2 file     will print only the second row in file (no body, print the row)

# AWK CAN DO ARITHMETICS
# without needing (()) let or bc
# awk '{total += $1} END {print total}' earnings.txt
# printf allows to print the result of a function, passed to the left as the value %
# the first argument "%.3f\n" sets floating point precision (.3f) to % and continues the otuput string
# awk 'BEGIN {printf "%.3f\n", 2005.50 / 3}'

# print, printf, sprintf
# print is the basic output function, automatically adding \n at the end and separating arguments with OFS (Output Field Separator, ' ' by default)
# printf is unforatted by default and has to be done manually: no \n at the end and no separations
# the syntax for printf is "formatted_string with % for each variable", variables in correct order
# the % can be followed by the type of the variable %s for strings %d int, %.1f for 1 digit floating point
# printf "%.3f\n", 2005.50 / 3         prints instead of % the result of the operation, 3 digits after the point, then new line
# printf "%s: %.2f\n", name, number    will print the value of name, followed by ': ' and then the value of number as 2 precision float, then newline
# sprintf has the same syntax as printf but returns a string with the same informations, instead of printing it

In [1]:
# EXAM EXERCISE

#!/bin/bash

cd $HOME #change working directory to $HOME
rm students/* #delete all elements int students/ if already present elements
rm -d students/ #delete the directory students, -d is needed
mkdir -p students #makes dir if not existent
if [ ! -f "./students/LCP_22-23_students.csv" ] #if not already present, as in if NOT -find PATH
then
    # wget imports the file from given url; the URL can be put inside '' or "" to prevent special symbols problems
    # nv prevents verbose, v allows more output; --tries limit the number of failure before error emerge
    # --directory-prefix changes the total path, downloading directly into subdirectories
    # -O redirect the output of the main file into a specified file_name, effectively changing the name of the downloaded file
    # ! -o and -O get different results, the first redirect the log output (so the verbose component) and only -O redirect the file content
    # wget -nv --tries=1 'https://www.dropbox.com/s/867rtx3az6e9gm8/LCP_22-23_students.csv' --directory-prefix='./students' -O LCP_22-23_students.csv
    wget -v --tries=1 https://www.dropbox.com/s/867rtx3az6e9gm8/LCP_22-23_students.csv --directory-prefix="./students" #import file in ./students
fi # if is composed of if, then, fi
cd students # pass to students directory
# touch is a good and safe measure to manage new files, evend tough most of commands are able to create a new file if needed
touch LCP_22-23_PoD_students.csv #create if not existing
touch LCP_22-23_Physics_students.csv #create if not existing
# grep returns lines in input file where element "PoD" is present, then this result is put into the second file
grep "PoD" LCP_22-23_students.csv > LCP_22-23_PoD_students.csv #copy only lines including PoD from file A to file B
grep "Physics" LCP_22-23_students.csv > LCP_22-23_Physics_students.csv #copy only lines including Physics from file A to file B
max=0 #setting a local variable (lowercase) to search maximum
max_L='A' #setting a local variable (lowercase) to search maximum corresponding letter
for i in {A..Z} #i assumes values in the alphabet capital letters
do
    # to j is assgined the return of the previous computed part inside `, the left part is executed then passed as input to the right part
    # for grep -v return the lines without the value, -e indicates which element is the selected value
    # -c perform a counting and ^ is used to indicate a stronger condition: the line has to start with the letter in i, not only contain that
    j=`grep -v -e  "^Family" LCP_22-23_students.csv | grep -c "^$i" LCP_22-23_students.csv` # first gives lines without Family Name (-v exludes -e is followed by the relative element), second counts starting with A
    # this is not actually required and makes a little mess for the output
    echo "$i : $j" #print out both variable; the $ is required in order to obtain the stored value, but works only inside double apices ""
    # -gt is a numerical boolean operation, so in order to be performed it has to go inside square brackets []
    if [ $j -gt $max ] #compare values, not string (> is for strings in bash, -gt is grater than for numbers)
    then
        max=$j #updating maximum
        max_L=$i #updating best letter
    fi
done #sintax for for is for do done
echo "Max surname starting letter $max_L with $max entries" # print found output
# counting all lines using the grep -c and searching for lines with '', so everyone
lines=`grep '' -c LCP_22-23_students.csv` #counts all lines in file
i=2 #starting from the third line (the file has a first empty line and then a second one made of metadata)
# a for can be used also
# for(( i=2; i<=lines; i++ ))
# notice that inside (( )) i and lines are automatically used by their value and not their name, and the operation < is performed as numerical
# the ambient (( )) allows numerical operation and casting of the variables
# another important note is that lines is the counting of the lines, so their exact number
# tecnically awk counts from 1 to N instead of the "usual" 0 to N-1
# inside the [] varibles have to pass by their value using $ and bool operations for numbers has to be done using -le instead of <=
while [ $i -le $lines ] #for each line, it will print it inside a different file, according to the modulo 18
do
    let g=($i-1)%18 #real computation for variables needs let or (( ))
    file="Group$g.csv" #naming scheme that uses value of g, there are ""
    touch $file #create file
    # awk does the same as cat or head but allows to select the exact line using NR, Number of Row; the output is redirected to file with >>
    awk NR==$i LCP_22-23_students.csv >> $file #selected line with awk
    let i+=1 #updating i using let to change the value and perform operations
done

SyntaxError: invalid decimal literal (971401427.py, line 9)

In [1]:
# 1.a Make a new directory called students in your home. Download a csv file with the list of students of this lab from here (use the wget command) and copy that to students. First check whether the file is already there

# 1.b Make two new files, one containing the students belonging to PoD, the other to Physics.

# 1.c For each letter of the alphabet, count the number of students whose surname starts with that letter.

# 1.d Find out which is the letter with most counts.

# 1.e Assume an obvious numbering of the students in the file (first line is 1, second line is 2, etc.), group students "modulo 18", i.e. 1,19,37,.. 2,20,38,.. etc. and put each group in a separate file

# !\bin\bash
# 1.a
cd $HOME
mkdir -p students
if [ ! -f "./students/LCP_22-23_students.csv" ]
then
    wget -nv  'https://www.dropbox.com/s/867rtx3az6e9gm8/LCP_22-23_students.csv' -O "./students/LCP_22-23_students.csv"
fi
# 1.b
cd students
touch 'LCP_22-23_PoD_students.csv'
touch 'LCP_22-23_Physics_students.csv'
grep -e "PoD" 'LCP_22-23_students.csv' > 'LCP_22-23_PoD_students.csv'
grep -e "Physics" 'LCP_22-23_students.csv' > 'LCP_22-23_Physics_students.csv'

# 1.c
max_c=0
max_L='A'
for i in {A..Z}
do
    c=$( grep -c -e "^$i" 'LCP_22-23_students.csv' )
# 1.d
    if (( c > max_c ))
    then
        max_c=$c
        max_L=$i
    fi
done
echo "The most used letter for the surname is $max_L with $max_c appearance"

# 1.e
lines=$( grep -c '' 'LCP_22-23_students.csv' )
for(( i=2; i<lines; i++ ))
do
        j=$(  echo "($i-1)%18" | bc )
        awk NR==$( echo "$i-1" | bc ) 'LCP_22-23_students.csv' >> "Group$j.txt"
done
# autodelete
rm *
cd ..
rm -d students


SyntaxError: invalid decimal literal (596507372.py, line 17)

### Ex 2

2.a Make a copy of the file `data.csv` removing the metadata and the commas between numbers; call it `data.txt`

2\.b How many even numbers are there?

2\.c Distinguish the entries on the basis of `sqrt(X^2 + Y^2 + Z^2)` is greater or smaller than `100*sqrt(3)/2`. Count the entries of each of the two groups 

2\.d Make `n` copies of data.txt (with `n` an input parameter of the script), where the i-th copy has all the numbers divided by i (with `1<=i<=n`).

In [ ]:
# EXAM EXERCISE

#!/bin/bash

# all lines that doesn't start with # (metadata) in data.csv are passed to sed
# sed is able to change using input element e 's/,//g' from , to nothing (/ separator define the fields, s is for inverting, g for globally)
grep -v '^#' data.csv | sed -e 's/,//g' > data.txt #take the lines in data.csv that don't start with #, substitute (sed) element (-e) from , to nothing
even=0 #starting to count even
for i in `cat data.txt` #for element in the dump of the file, so the single number can be selected
do
    #IN BASH TRUE IS 0 AND 1 IS FALSE, [ ] make it interpret as bool, but (( )) is for numbers,  then converted to bools that way
    # inside (( )) the $ is not necessary and all variables are already used by their value
    if (( i%2 )) #%2 gives 0 if even, that correspond to true in bash
    then
        let even+=1 #update counter, let is mandatory if not inside (()); againt, using let let us operate directly on values without $
    fi
done
echo "even numbers = $even"
# setting new local variable to count
m=0
l=0
# getting a set value for sigma, $ pass the value instead of the command, |bc make it compute in floating point operation, '' are ok because there are no variables
# $ before the ( ) grants that the operation in () is performed as command and then passed as its value to the variable to which is assigned
sigma=$( echo 'scale=6;100*sqrt(3)/2.0' | bc)
FILE="data.txt" #global variable FILE
lines=`grep '' -c $FILE` #counts all lines in file
i=1 #starts from second line
while [ $i -le $lines ] #cycle over number of lines
do
    # note that NR and other parameter requires the == to assign them the value when calling the command
    line=`awk NR==$i $FILE` #line is the string of the single i line in FILE
    # <<< uses heredoc << with < to give the object line as an input
    # this could also have been done using read -a p <<< "$line", so that p is an array of elements X,Y,Z,x,y,z
    # IFS is the Internal Field Separator, indicating how different words are separated from each others; the default is ' ' so here is useless
    IFS=' ' read -r X Y Z x y z <<< "$line" #set variables X, Y, Z, x, y, z from the line using separator ' ', read requires <<<
    # to the variable d is assigned a value computed by command substitution, where echo passes a string of operations to bc
    d=$( echo "scale=6;sqrt($X*$X+$Y*$Y+$Z*$Z)" | bc) # assign to d the distance computed
    # echo $d
    if [ `echo "$d < $sigma" | bc` -eq 0 ] #this is a bool op. because of -eq, check if d<=sigma, using bc to do calculation where < is ok
    then
        let m++ #update more than
    else
        let l++ #update less than
    fi
    let i++ #update i
done
echo "there are $m of distance grater than $sigma"
echo "there are $l of distance smaller than $sigma"
if [ -z $1 ] #1 is the first input when calling the script, -z check if it is a NULL, apparently -z is bool op
then
    echo "This program requires an input for normalization"
    exit
fi
if [ $1 -lt 1 ] #check the normalization if <= 1
then
    echo "This program requires an input grater than 1 for normalization"
    exit
fi
for (( i=1; i<=$1; i++ )) #computing operation (())
do
    # DIFFERENCE -v for awk is value, for grep is "without"
    #-v passes i as a variable to awk; cycle over NF Number of Fields; $j content of j field, 
    #check if field equal to number and end ($); then print as float j/i
    awk -v i="$i" '{for(j=1;j<=NF;j++) if($j~/^[0-9]+$/) $j=sprintf("%.1f",$j/i)}1' data.txt > "data$i.csv"
    # this can be written like
    # awk -v i="$i" '                       # awk called with a parametric value i and input the file
    # {
    #     for(j=1;j<=NF;j++) {              # cycling over all the input elements or fields (NF is Number of Fields)
    #         if($j~/^[0-9]+$/) {           # this checks only if j is a numerical integer, it is not necessary and also works only in N
    #             $j=sprintf("%.1f",$j/i)   # this executes the desired operation
    #         }
    #     }
    #     print
    # } ' data.txt > data"$i".csv           # redirected output
    done
#[0-9]* for zero or more numbers; [0-9][0-9]* one or more; etc. those are patterns with 2 and 3 element resp. \1 \2 \3
# % echo "123 abc" | sed 's/[0-9]*/& &/'
# 123 123 abc
# sed y is like tr

In [ ]:
# 2.a Make a copy of the file data.csv removing the metadata and the commas between numbers; call it data.txt

# 2.b How many even numbers are there?

# 2.c Distinguish the entries on the basis of sqrt(X^2 + Y^2 + Z^2) is greater or smaller than 100*sqrt(3)/2. Count the entries of each of the two groups

# 2.d Make n copies of data.txt (with n an input parameter of the script), where the i-th copy has all the numbers divided by i (with 1<=i<=n).


# 2.a
touch data.txt
grep -v "^#" "data.csv" | sed -e 's/,//g' > data.txt

# 2.b
even=0
for i in $( < data.txt )
do
    if (( i%2 ))
    then
        let even++
    fi
done
echo "There are $even even numbers"

# 2.c
sigma=$( echo ' scale=5;100.0*sqrt(3.0)/2.0' | bc )
lines=$( grep -c '' "data.txt" )
less=0
more=0
for i in $( seq 1 $lines )
do
    line=$( awk NR==$i "data.txt" )
    #read -a pos <<< $line
    read -r X Y Z x y z <<< "$line"
    #distance=$( echo "sqrt(${pos[0]}**2 + ${pos[1]}**2 + ${pos[2]}**2)" | bc )
    distance=$( echo " scale=5; sqrt($X*$X + $Y*$Y + $Z*$Z)" | bc )
    if  (( $( echo "$distance < $sigma" | bc ) ))
    then
        let more++
    else
        let less++
    fi
done
echo "There are $less smaller than $sigma and $more grater"

# 2.d
if [ -z $1 ]
then
    echo "This program requires an input"
    exit
fi
if [ $1 -lt 1 ]
then
    echo "This program requires an input grater than 1"
    exit
fi    
n=$1
i=1
while [ $i -lt $n ]
do
    awk -v i=$i '{
        for(j=1;j<=NF;j++){
            $j=sprintf("%.1f",$j/i)
            }
        print $0
        }' "data.txt" > "data$i.csv"
    let i++
done